
# PCA vs ICA vs MS-AA: method-calibrated network-motif simulation benchmark

This notebook compares PCA, ICA, and multisubject archetypal analysis (MS-AA) across four core generative regimes:

1. **PCA-favorable control** — orthogonal latent spatial factors with independent Gaussian scores and distinct variances.
2. **ICA-favorable control** — nonorthogonal linear mixtures of statistically independent, non-Gaussian latent sources.
3. **AA-favorable control** — sparse convex mixtures of nonorthogonal extremal network motifs.
4. **Brain-inspired metastable dynamics** — recurring, overlapping network configurations with temporal persistence, gradual transitions, low-level coactivation, subject variability, and measurement noise.

The purpose is method calibration rather than universal method ranking: each method should perform best in the regime most closely aligned with its assumptions, and the metastable regime is the scientifically important comparison.

## ICA policy

To keep the benchmark simple and complete, ICA is fit once per simulated dataset using a deterministic random seed.

- ICA metrics are **always retained**, even if FastICA emits a convergence warning.
- Convergence status and iteration count are reported as diagnostics.
- Nonconvergence is therefore visible rather than converted into missing values.
- In the Gaussian PCA-favorable control, ICA nonconvergence is not surprising because independent Gaussian components are not uniquely identifiable; the recovery estimate can still be shown descriptively with the convergence flag.

For `sklearn.decomposition.FastICA`, the spatial patterns corresponding to the generative mixing vectors are the columns of `mixing_`. Therefore the notebook compares the true spatial patterns with `ica.mixing_.T`, not `ica.components_`.

All sign-indeterminate methods (PCA and ICA) are evaluated using absolute spatial pattern correlations.


In [ ]:

# ============================================================
# Configuration
# ============================================================
import numpy as np
RUN_MODE = "paper"   # "quick" or "paper"
RNG_SEED = 42

HELPERS_DIR = "helpers"
OUTPUT_DIR = "simulation_pca_ica_msaa_outputs_simpleica"

ANALYSIS_TYPE = "spatial"

MSAA_OPTS_BASE = dict(
    maxiter=200,
    conv_crit=1e-6,
    fix_var_iter=5,
    heteroscedastic=False,
    use_gpu=False,
    rngSEED=RNG_SEED,
)

# Shared spatial architecture
N_NETWORKS = 6
SUBNETWORKS_PER_NETWORK = 2
NODES_PER_SUBNETWORK = 10
N_TRUE_MOTIFS = 6

BASE_NOISE_SD = 0.25
BASE_MULTIPLICATIVE_SD = 0.08
BASE_ADDITIVE_SD = 0.04

# PCA-positive-control source standard deviations.
# Distinct variances make the PCA axes identifiable.
PCA_SOURCE_SCALES = np.array([2.0, 1.7, 1.4, 1.1, 0.8, 0.5]) if False else None

# AA-positive-control geometry
DIRICHLET_ALPHA = 0.10

# Brain-inspired metastable dynamics
MEAN_DWELL = 12
TRANSITION_WIDTH = 4
SECONDARY_WEIGHT = 0.25
AR1_RHO = 0.85

# Secondary robustness
NOISE_LEVELS = [0.10, 0.25, 0.40]
HETEROGENEITY_LEVELS = [0.0, 1.0, 1.5]
K_LEVELS = [4, 6, 8]

if RUN_MODE == "quick":
    N_SUBJECTS = 10
    T_PER_SUBJECT = 120
    N_REPS_CORE = 2
    N_REPS_METASTABLE = 3
    N_REPS_ROBUST = 1
elif RUN_MODE == "paper":
    N_SUBJECTS = 16
    T_PER_SUBJECT = 180
    N_REPS_CORE = 20
    N_REPS_METASTABLE = 20
    N_REPS_ROBUST = 10
else:
    raise ValueError("RUN_MODE must be 'quick' or 'paper'")

PCA_SOURCE_SCALES = np.array([2.0, 1.7, 1.4, 1.1, 0.8, 0.5])

print("RUN_MODE:", RUN_MODE)
print("Subjects:", N_SUBJECTS)
print("Time points per subject:", T_PER_SUBJECT)


In [ ]:

# ============================================================
# Imports
# ============================================================

%matplotlib inline

import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA, FastICA
from sklearn.exceptions import ConvergenceWarning

sys.path.append(HELPERS_DIR)
from MultiSubject_AA import Subject, multi_subject_aa

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = OUTPUT_DIR / f"pca_ica_msaa_results_{RUN_MODE}.csv"

print("Checkpoint:", RESULTS_CSV)


## 1. MS-AA helpers matching the empirical pipeline

In [ ]:

def coerce_TxV(x):
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D T x V matrix, got {x.shape}")

    x = x - x.mean(axis=0, keepdims=True)
    return x.astype(np.float32, copy=False)


def orient_subject_matrix(X_raw, analysis_type):
    Xtv = coerce_TxV(X_raw)
    return Xtv if analysis_type == "spatial" else Xtv.T


def build_subject_objects(subject_list, analysis_type):
    return [
        Subject(
            X=orient_subject_matrix(x, analysis_type),
            sX=orient_subject_matrix(x, analysis_type),
        )
        for x in subject_list
    ]


def fit_msaa(subject_list, K, analysis_type="spatial", seed=RNG_SEED, verbose=False):
    opts = dict(MSAA_OPTS_BASE)
    opts["rngSEED"] = int(seed)

    subj_objs = build_subject_objects(subject_list, analysis_type)

    t0 = time.time()

    results_subj, C, cost_fun, varexpl, secs = multi_subject_aa(
        subj_objs,
        noc=int(K),
        opts=opts,
    )

    elapsed = time.time() - t0

    if verbose:
        print(
            f"[MS-AA] K={K} | VarExpl={varexpl*100:.2f}% | "
            f"{elapsed:.2f}s"
        )

    return {
        "results_subj": results_subj,
        "C": C,
        "cost_fun": cost_fun,
        "varexpl": varexpl,
        "secs": secs,
        "elapsed": elapsed,
        "K": int(K),
    }


def reconstruct_from_archetypes(sub, analysis_type="spatial"):
    sXC = np.asarray(sub["sXC"], dtype=float)
    S = np.asarray(sub["S"], dtype=float)

    if analysis_type == "spatial":
        return sXC @ S

    return (sXC @ S).T


## 2. Network-motif generator

In [ ]:

def make_overlapping_motifs():
    n_subnetworks = N_NETWORKS * SUBNETWORKS_PER_NETWORK
    V = n_subnetworks * NODES_PER_SUBNETWORK

    motifs = np.full(
        (N_TRUE_MOTIFS, V),
        0.05,
        dtype=float,
    )

    motif_subnetworks = [
        [0, 3, 8],
        [1, 3, 9],
        [2, 5, 10],
        [2, 4, 11],
        [0, 6, 7],
        [1, 6, 8],
    ]

    for k, active in enumerate(motif_subnetworks):
        for subnet in active:
            lo = subnet * NODES_PER_SUBNETWORK
            hi = lo + NODES_PER_SUBNETWORK
            motifs[k, lo:hi] = 1.0

    return motifs


TRUE_MOTIFS_RAW = make_overlapping_motifs()
V = TRUE_MOTIFS_RAW.shape[1]

print("Motif matrix:", TRUE_MOTIFS_RAW.shape)


In [ ]:

def plot_pattern_matrix(patterns, title, ylabel="Pattern"):
    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.imshow(patterns, aspect="auto")

    for x in range(
        NODES_PER_SUBNETWORK,
        patterns.shape[1],
        NODES_PER_SUBNETWORK,
    ):
        ax.axvline(x - 0.5, linewidth=0.4, alpha=0.35)

    network_width = (
        SUBNETWORKS_PER_NETWORK
        * NODES_PER_SUBNETWORK
    )

    for x in range(
        network_width,
        patterns.shape[1],
        network_width,
    ):
        ax.axvline(x - 0.5, linewidth=1.2)

    ax.set_xlabel("Nodes grouped by network → subnetwork")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)

    plt.show()


plot_pattern_matrix(
    TRUE_MOTIFS_RAW,
    "Ground-truth overlapping distributed network motifs",
    ylabel="True motif",
)


## 3. Common normalization and evaluation metrics

In [ ]:

def normalize_across_subjects(subject_list):
    subjects = [
        np.asarray(x, dtype=float)
        for x in subject_list
    ]

    stacked = np.vstack(subjects)

    mu = stacked.mean(axis=0, keepdims=True)
    sd = stacked.std(axis=0, keepdims=True) + 1e-8

    normalized = [
        ((x - mu) / sd).astype(
            np.float32,
            copy=False,
        )
        for x in subjects
    ]

    return normalized, mu, sd


def pattern_corr_matrix(A, B, absolute=False):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    A0 = A - A.mean(axis=1, keepdims=True)
    B0 = B - B.mean(axis=1, keepdims=True)

    A0 /= (
        np.linalg.norm(A0, axis=1, keepdims=True)
        + 1e-12
    )

    B0 /= (
        np.linalg.norm(B0, axis=1, keepdims=True)
        + 1e-12
    )

    C = A0 @ B0.T

    return np.abs(C) if absolute else C


def optimal_match(
    true_patterns,
    recovered_patterns,
    absolute=False,
):
    C = pattern_corr_matrix(
        true_patterns,
        recovered_patterns,
        absolute=absolute,
    )

    true_idx, recovered_idx = linear_sum_assignment(-C)

    return {
        "similarity": C,
        "true_idx": true_idx,
        "recovered_idx": recovered_idx,
        "matched": C[true_idx, recovered_idx],
    }


def coverage_adjusted_recovery(match, n_true):
    # Penalizes K < number of true patterns.
    return float(
        np.sum(match["matched"]) / n_true
    )


def collapse_to_subnetworks(patterns):
    patterns = np.asarray(patterns, dtype=float)

    n_subnetworks = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
    )

    out = np.zeros(
        (patterns.shape[0], n_subnetworks)
    )

    for subnet in range(n_subnetworks):
        lo = subnet * NODES_PER_SUBNETWORK
        hi = lo + NODES_PER_SUBNETWORK
        out[:, subnet] = patterns[:, lo:hi].mean(axis=1)

    return out


def dimension_mixing_metrics(
    true_patterns,
    recovered_patterns,
):
    C = pattern_corr_matrix(
        recovered_patterns,
        true_patterns,
        absolute=True,
    )

    squared = C ** 2

    probs = squared / (
        squared.sum(axis=1, keepdims=True)
        + 1e-12
    )

    entropy = -np.sum(
        probs * np.log(probs + 1e-12),
        axis=1,
    )

    entropy /= np.log(true_patterns.shape[0])

    purity = 1.0 - entropy

    sorted_corr = np.sort(C, axis=1)[:, ::-1]

    top1 = sorted_corr[:, 0]
    top2 = (
        sorted_corr[:, 1]
        if sorted_corr.shape[1] > 1
        else np.zeros_like(top1)
    )

    return {
        "purity": float(np.mean(purity)),
        "selectivity": float(
            np.mean(top1 - top2)
        ),
        "mixing_ratio": float(
            np.mean(
                top2 / (top1 + 1e-12)
            )
        ),
    }


def linear_span_recovery(
    true_patterns,
    recovered_patterns,
):
    # Least-squares projection of each true pattern
    # onto the row span of the recovered patterns.
    B = np.asarray(recovered_patterns, dtype=float).T

    vals = []

    for true in np.asarray(true_patterns, dtype=float):
        coef, *_ = np.linalg.lstsq(
            B,
            true,
            rcond=None,
        )

        recon = B @ coef

        vals.append(
            np.corrcoef(true, recon)[0, 1]
        )

    return float(np.nanmean(vals))



## 4. Unified PCA / ICA / MS-AA evaluator

The direct comparison is between **spatial patterns**:

- PCA: `pca.components_`
- ICA: `ica.mixing_.T`
- MS-AA: group-averaged `S`

PCA and ICA pattern correlations are sign-invariant. MS-AA uses signed correlations because its archetypes are not sign-indeterminate in the same way.

For fitted \(K < K_{\mathrm{true}}\), the notebook also reports a **coverage-adjusted recovery** score so that recovering only a subset of true motifs cannot look artificially strong.


In [ ]:

def fit_fastica(X_stack, K_fit, seed):
    ica = FastICA(
        n_components=int(K_fit),
        whiten="unit-variance",
        algorithm="parallel",
        fun="logcosh",
        max_iter=3000,
        tol=1e-5,
        random_state=int(seed),
    )

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter(
            "always",
            ConvergenceWarning,
        )

        Z = ica.fit_transform(X_stack)
        Xhat = ica.inverse_transform(Z)

    converged = not any(
        issubclass(
            w.category,
            ConvergenceWarning,
        )
        for w in caught
    )

    spatial_patterns = np.asarray(
        ica.mixing_,
        dtype=float,
    ).T

    return {
        "model": ica,
        "scores": Z,
        "reconstruction": Xhat,
        "spatial_patterns": spatial_patterns,
        "converged": bool(converged),
        "n_iter": int(
            getattr(
                ica,
                "n_iter_",
                -1,
            )
        ),
    }


def evaluate_methods(
    subject_data,
    true_patterns_raw,
    K_fit,
    seed,
    regime,
    rep,
    extra=None,
    verbose=False,
):
    subjects, mu, sd = normalize_across_subjects(
        subject_data
    )

    true_patterns = (
        true_patterns_raw - mu
    ) / sd

    X_stack = np.vstack(subjects)

    # ========================================================
    # MS-AA
    # ========================================================

    msaa = fit_msaa(
        subjects,
        K=K_fit,
        analysis_type="spatial",
        seed=seed,
        verbose=verbose,
    )

    S_group = np.mean(
        np.stack(
            [
                np.asarray(
                    sub["S"],
                    dtype=float,
                )
                for sub in msaa["results_subj"]
            ],
            axis=0,
        ),
        axis=0,
    )

    Xhat_aa = np.vstack([
        reconstruct_from_archetypes(
            sub,
            "spatial",
        )
        for sub in msaa["results_subj"]
    ])

    aa_mse = float(
        np.mean(
            (X_stack - Xhat_aa) ** 2
        )
    )

    # ========================================================
    # PCA
    # ========================================================

    pca = PCA(
        n_components=int(K_fit),
        random_state=int(seed),
    )

    pca_scores = pca.fit_transform(X_stack)
    Xhat_pca = pca.inverse_transform(pca_scores)

    pca_patterns = np.asarray(
        pca.components_,
        dtype=float,
    )

    pca_mse = float(
        np.mean(
            (X_stack - Xhat_pca) ** 2
        )
    )

    # ========================================================
    # ICA
    # ========================================================

    ica_fit = fit_fastica(
        X_stack,
        K_fit,
        seed,
    )

    ica_patterns = ica_fit[
        "spatial_patterns"
    ]

    Xhat_ica = ica_fit[
        "reconstruction"
    ]

    ica_mse = float(
        np.mean(
            (X_stack - Xhat_ica) ** 2
        )
    )

    # ========================================================
    # Direct spatial-pattern recovery
    # ========================================================

    aa_match = optimal_match(
        true_patterns,
        S_group,
        absolute=False,
    )

    pca_match = optimal_match(
        true_patterns,
        pca_patterns,
        absolute=True,
    )

    ica_match = optimal_match(
        true_patterns,
        ica_patterns,
        absolute=True,
    )

    # ========================================================
    # Subnetwork-level recovery
    # ========================================================

    true_sub = collapse_to_subnetworks(
        true_patterns
    )

    aa_sub = collapse_to_subnetworks(
        S_group
    )

    pca_sub = collapse_to_subnetworks(
        pca_patterns
    )

    ica_sub = collapse_to_subnetworks(
        ica_patterns
    )

    aa_sub_match = optimal_match(
        true_sub,
        aa_sub,
        absolute=False,
    )

    pca_sub_match = optimal_match(
        true_sub,
        pca_sub,
        absolute=True,
    )

    ica_sub_match = optimal_match(
        true_sub,
        ica_sub,
        absolute=True,
    )

    # ========================================================
    # Span recovery
    # ========================================================

    aa_span = linear_span_recovery(
        true_patterns,
        S_group,
    )

    pca_span = linear_span_recovery(
        true_patterns,
        pca_patterns,
    )

    ica_span = linear_span_recovery(
        true_patterns,
        ica_patterns,
    )

    # ========================================================
    # Mixing diagnostics
    # ========================================================

    aa_mix = dimension_mixing_metrics(
        true_patterns,
        S_group,
    )

    pca_mix = dimension_mixing_metrics(
        true_patterns,
        pca_patterns,
    )

    ica_mix = dimension_mixing_metrics(
        true_patterns,
        ica_patterns,
    )

    n_true = true_patterns.shape[0]

    row = {
        "regime": regime,
        "condition": "base",
        "rep": int(rep),
        "seed": int(seed),
        "K_fit": int(K_fit),

        "aa_motif_recovery": float(
            np.mean(
                aa_match["matched"]
            )
        ),
        "pca_motif_recovery": float(
            np.mean(
                pca_match["matched"]
            )
        ),
        "ica_motif_recovery": float(
            np.mean(
                ica_match["matched"]
            )
        ),

        "aa_motif_recovery_coverage": (
            coverage_adjusted_recovery(
                aa_match,
                n_true,
            )
        ),
        "pca_motif_recovery_coverage": (
            coverage_adjusted_recovery(
                pca_match,
                n_true,
            )
        ),
        "ica_motif_recovery_coverage": (
            coverage_adjusted_recovery(
                ica_match,
                n_true,
            )
        ),

        "aa_subnetwork_recovery": float(
            np.mean(
                aa_sub_match["matched"]
            )
        ),
        "pca_subnetwork_recovery": float(
            np.mean(
                pca_sub_match["matched"]
            )
        ),
        "ica_subnetwork_recovery": float(
            np.mean(
                ica_sub_match["matched"]
            )
        ),

        "aa_span_recovery": aa_span,
        "pca_span_recovery": pca_span,
        "ica_span_recovery": ica_span,

        "aa_mse": aa_mse,
        "pca_mse": pca_mse,
        "ica_mse": ica_mse,

        "aa_purity": aa_mix["purity"],
        "pca_purity": pca_mix["purity"],
        "ica_purity": ica_mix["purity"],

        "aa_selectivity": aa_mix[
            "selectivity"
        ],
        "pca_selectivity": pca_mix[
            "selectivity"
        ],
        "ica_selectivity": ica_mix[
            "selectivity"
        ],

        "aa_mixing_ratio": aa_mix[
            "mixing_ratio"
        ],
        "pca_mixing_ratio": pca_mix[
            "mixing_ratio"
        ],
        "ica_mixing_ratio": ica_mix[
            "mixing_ratio"
        ],

        "aa_varexpl": float(
            msaa["varexpl"]
        ),
        "pca_explained_variance": float(
            np.sum(
                pca.explained_variance_ratio_
            )
        ),

        # Keep ICA convergence as a diagnostic only.
        "ica_converged": bool(
            ica_fit["converged"]
        ),
        "ica_n_iter": int(
            ica_fit["n_iter"]
        ),

        "msaa_elapsed_sec": float(
            msaa["elapsed"]
        ),
    }

    if extra:
        row.update(extra)

    return row



# REGIME 1 — PCA-favorable control

The true spatial factors are orthogonal. Their temporal scores are independent Gaussian variables with **distinct variances**, making the generating PCA axes identifiable in the population covariance.

PCA should be the strongest direct-pattern method here.

ICA has no unique independent-component solution for Gaussian sources, while MS-AA is being asked to represent a variance-axis geometry that is not archetypal.


In [ ]:

def simulate_pca_control(seed):
    rng = np.random.default_rng(seed)

    Q, _ = np.linalg.qr(
        rng.normal(
            size=(V, N_TRUE_MOTIFS)
        )
    )

    true_factors = Q.T

    subjects = []

    for s in range(N_SUBJECTS):
        scores = rng.normal(
            size=(
                T_PER_SUBJECT,
                N_TRUE_MOTIFS,
            )
        )

        scores = (
            scores
            * PCA_SOURCE_SCALES[None, :]
        )

        X = scores @ true_factors

        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)

    return subjects, true_factors



# REGIME 2 — ICA-favorable control

The true spatial patterns are dense and nonorthogonal. Their temporal sources are statistically independent and strongly non-Gaussian (Laplace), and observations are generated by a linear mixing model.

ICA should recover these spatial mixing patterns most directly.

PCA can capture their low-dimensional span but is not expected to identify the independent mixing vectors. MS-AA is not being given an extremal/simplex geometry.


In [ ]:

def make_ica_mixing_patterns(seed):
    rng = np.random.default_rng(seed)

    # Start from random dense vectors and add a modest shared
    # direction so the mixing patterns are nonorthogonal.
    base = rng.normal(
        size=(N_TRUE_MOTIFS, V)
    )

    shared = rng.normal(size=(1, V))
    shared /= (
        np.linalg.norm(shared) + 1e-12
    )

    patterns = (
        base
        + 0.45 * shared
    )

    patterns -= patterns.mean(
        axis=1,
        keepdims=True,
    )

    patterns /= (
        np.linalg.norm(
            patterns,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return patterns


def simulate_ica_control(seed):
    rng = np.random.default_rng(seed)

    true_patterns = make_ica_mixing_patterns(
        seed + 999
    )

    subjects = []

    for s in range(N_SUBJECTS):
        # Independent, super-Gaussian latent sources.
        sources = rng.laplace(
            loc=0.0,
            scale=1.0 / np.sqrt(2.0),
            size=(
                T_PER_SUBJECT,
                N_TRUE_MOTIFS,
            ),
        )

        X = sources @ true_patterns

        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)

    return subjects, true_patterns



# REGIME 3 — AA-favorable control

Each observation is a sparse convex mixture of nonorthogonal extremal motifs, with mild subject-specific perturbations.

MS-AA should recover the generating motifs most directly.

PCA and ICA may still preserve much of the low-dimensional span, but their individual components need not correspond to the extremal generators.


In [ ]:

def simulate_aa_control(seed):
    rng = np.random.default_rng(seed)

    subjects = []

    for s in range(N_SUBJECTS):
        multiplicative = (
            1.0
            + rng.normal(
                0,
                BASE_MULTIPLICATIVE_SD,
                size=TRUE_MOTIFS_RAW.shape,
            )
        )

        additive = rng.normal(
            0,
            BASE_ADDITIVE_SD,
            size=TRUE_MOTIFS_RAW.shape,
        )

        subj_motifs = (
            TRUE_MOTIFS_RAW
            * multiplicative
            + additive
        )

        W = rng.dirichlet(
            np.full(
                N_TRUE_MOTIFS,
                DIRICHLET_ALPHA,
            ),
            size=T_PER_SUBJECT,
        )

        X = W @ subj_motifs

        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)

    return subjects, TRUE_MOTIFS_RAW



# REGIME 4 — Brain-inspired metastable network dynamics

This is deliberately **not** a PCA, ICA, or AA generative model.

Network configurations recur over time, remain active for variable dwell periods, transition gradually, partially coactivate, and vary across subjects. Background motif recruitment follows a temporally autocorrelated process.

\[
X_t = \sum_k A_{tk}M_k^{(s)} + \epsilon_t.
\]

The latent activation matrix \(A\) is generated by a persistent dynamical process rather than by independent Gaussian factors, independent non-Gaussian sources, or independent Dirichlet draws.


In [ ]:

def sample_state_sequence(
    T,
    K,
    rng,
    mean_dwell=MEAN_DWELL,
):
    states = []
    current = int(rng.integers(K))

    while len(states) < T:
        dwell = max(
            2,
            int(
                rng.geometric(
                    1.0 / mean_dwell
                )
            ),
        )

        states.extend(
            [current] * dwell
        )

        candidates = [
            k
            for k in range(K)
            if k != current
        ]

        current = int(
            rng.choice(candidates)
        )

    return np.asarray(
        states[:T],
        dtype=int,
    )


def smooth_state_activations(
    states,
    K,
    transition_width=TRANSITION_WIDTH,
):
    T = len(states)

    A = np.zeros(
        (T, K),
        dtype=float,
    )

    A[
        np.arange(T),
        states,
    ] = 1.0

    if transition_width > 0:
        kernel_x = np.arange(
            -transition_width,
            transition_width + 1,
        )

        sigma = max(
            1.0,
            transition_width / 2,
        )

        kernel = np.exp(
            -(kernel_x ** 2)
            / (2 * sigma ** 2)
        )

        kernel /= kernel.sum()

        for k in range(K):
            A[:, k] = np.convolve(
                A[:, k],
                kernel,
                mode="same",
            )

    A /= (
        A.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return A


def add_correlated_secondary_recruitment(
    A,
    rng,
):
    T, K = A.shape

    background = np.zeros(
        (T, K),
        dtype=float,
    )

    eps = rng.normal(
        0,
        1,
        size=(T, K),
    )

    for t in range(1, T):
        background[t] = (
            AR1_RHO
            * background[t - 1]
            + np.sqrt(
                1 - AR1_RHO ** 2
            )
            * eps[t]
        )

    background = np.exp(background)

    background /= (
        background.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    mixed = (
        (1.0 - SECONDARY_WEIGHT) * A
        + SECONDARY_WEIGHT * background
    )

    mixed /= (
        mixed.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return mixed


def simulate_metastable(
    seed,
    noise_sd=BASE_NOISE_SD,
    heterogeneity_scale=1.0,
):
    rng = np.random.default_rng(seed)

    subjects = []
    state_sequences = []
    activations = []

    for s in range(N_SUBJECTS):
        multiplicative = (
            1.0
            + rng.normal(
                0,
                BASE_MULTIPLICATIVE_SD
                * heterogeneity_scale,
                size=TRUE_MOTIFS_RAW.shape,
            )
        )

        additive = rng.normal(
            0,
            BASE_ADDITIVE_SD
            * heterogeneity_scale,
            size=TRUE_MOTIFS_RAW.shape,
        )

        subj_motifs = (
            TRUE_MOTIFS_RAW
            * multiplicative
            + additive
        )

        states = sample_state_sequence(
            T_PER_SUBJECT,
            N_TRUE_MOTIFS,
            rng,
        )

        A_state = smooth_state_activations(
            states,
            N_TRUE_MOTIFS,
        )

        A = (
            add_correlated_secondary_recruitment(
                A_state,
                rng,
            )
        )

        X = A @ subj_motifs

        X += rng.normal(
            0,
            noise_sd,
            size=X.shape,
        )

        subjects.append(X)
        state_sequences.append(states)
        activations.append(A)

    return {
        "subjects": subjects,
        "true_patterns": TRUE_MOTIFS_RAW,
        "states": state_sequences,
        "activations": activations,
    }


## 5. Metastable-process diagnostics

In [ ]:

def metastable_diagnostics(sim):
    dwell_lengths = []

    for states in sim["states"]:
        start = 0

        for t in range(
            1,
            len(states) + 1,
        ):
            if (
                t == len(states)
                or states[t]
                != states[start]
            ):
                dwell_lengths.append(
                    t - start
                )
                start = t

    lag1 = []

    for A in sim["activations"]:
        for k in range(A.shape[1]):
            lag1.append(
                np.corrcoef(
                    A[:-1, k],
                    A[1:, k],
                )[0, 1]
            )

    return {
        "mean_dwell": float(
            np.mean(dwell_lengths)
        ),
        "median_dwell": float(
            np.median(dwell_lengths)
        ),
        "mean_activation_lag1": float(
            np.nanmean(lag1)
        ),
    }


demo = simulate_metastable(RNG_SEED)

print(
    metastable_diagnostics(demo)
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(demo["activations"][0])
ax.set_xlabel("Time")
ax.set_ylabel("Motif activation")
ax.set_title(
    "Example metastable latent activation trajectories"
)
plt.show()


## 6. Restart-safe checkpoints

In [ ]:

def load_results():
    if RESULTS_CSV.exists():
        df = pd.read_csv(RESULTS_CSV)
        print(
            f"Loaded {len(df)} completed simulations"
        )
        return df

    return pd.DataFrame()


def is_complete(
    df,
    regime,
    condition,
    rep,
):
    if df.empty:
        return False

    mask = (
        (df["regime"] == regime)
        & (df["condition"] == condition)
        & (
            df["rep"].astype(int)
            == int(rep)
        )
    )

    return bool(mask.any())


def save_row(df, row):
    new = pd.DataFrame([row])

    out = (
        new
        if df.empty
        else pd.concat(
            [df, new],
            ignore_index=True,
        )
    )

    out = out.drop_duplicates(
        subset=[
            "regime",
            "condition",
            "rep",
        ],
        keep="last",
    )

    tmp = RESULTS_CSV.with_suffix(
        ".tmp.csv"
    )

    out.to_csv(
        tmp,
        index=False,
    )

    os.replace(
        tmp,
        RESULTS_CSV,
    )

    return out


# PART A — Four-regime core benchmark

In [ ]:

results_df = load_results()

core_specs = [
    (
        "PCA_control",
        simulate_pca_control,
        N_REPS_CORE,
        101,
    ),
    (
        "ICA_control",
        simulate_ica_control,
        N_REPS_CORE,
        151,
    ),
    (
        "AA_control",
        simulate_aa_control,
        N_REPS_CORE,
        202,
    ),
]

for (
    regime,
    generator,
    n_reps,
    regime_code,
) in core_specs:

    for rep in range(n_reps):
        condition = "base"

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {regime} rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    regime_code,
                    rep,
                ]
            ).generate_state(1)[0]
        )

        subjects, true_patterns = (
            generator(seed)
        )

        row = evaluate_methods(
            subjects,
            true_patterns,
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            extra={
                "condition": condition
            },
        )

        results_df = save_row(
            results_df,
            row,
        )


# Metastable
for rep in range(N_REPS_METASTABLE):
    regime = "metastable"
    condition = "base"

    if is_complete(
        results_df,
        regime,
        condition,
        rep,
    ):
        print(
            f"SKIP {regime} rep={rep}"
        )
        continue

    seed = int(
        np.random.SeedSequence(
            [
                RNG_SEED,
                303,
                rep,
            ]
        ).generate_state(1)[0]
    )

    sim = simulate_metastable(seed)
    diag = metastable_diagnostics(sim)

    row = evaluate_methods(
        sim["subjects"],
        sim["true_patterns"],
        K_fit=N_TRUE_MOTIFS,
        seed=seed,
        regime=regime,
        rep=rep,
        extra={
            "condition": condition,
            "mean_dwell": (
                diag["mean_dwell"]
            ),
            "mean_activation_lag1": (
                diag[
                    "mean_activation_lag1"
                ]
            ),
        },
    )

    results_df = save_row(
        results_df,
        row,
    )

print("Core four-regime benchmark complete.")


## 7. Core summary

In [ ]:

core = results_df[
    results_df["regime"].isin(
        [
            "PCA_control",
            "ICA_control",
            "AA_control",
            "metastable",
        ]
    )
].copy()

summary_cols = [
    "aa_motif_recovery",
    "pca_motif_recovery",
    "ica_motif_recovery",
    "aa_subnetwork_recovery",
    "pca_subnetwork_recovery",
    "ica_subnetwork_recovery",
    "aa_span_recovery",
    "pca_span_recovery",
    "ica_span_recovery",
    "aa_mse",
    "pca_mse",
    "ica_mse",
]

summary = (
    core
    .groupby("regime")[
        summary_cols
    ]
    .agg(["mean", "sem"])
)

display(summary)


In [ ]:

# Compact manuscript-oriented means table

compact = (
    core
    .groupby("regime")
    .agg(
        n=("rep", "count"),

        AA_recovery=(
            "aa_motif_recovery",
            "mean",
        ),
        PCA_recovery=(
            "pca_motif_recovery",
            "mean",
        ),
        ICA_recovery=(
            "ica_motif_recovery",
            "mean",
        ),

        AA_subnetwork=(
            "aa_subnetwork_recovery",
            "mean",
        ),
        PCA_subnetwork=(
            "pca_subnetwork_recovery",
            "mean",
        ),
        ICA_subnetwork=(
            "ica_subnetwork_recovery",
            "mean",
        ),

        AA_span=(
            "aa_span_recovery",
            "mean",
        ),
        PCA_span=(
            "pca_span_recovery",
            "mean",
        ),
        ICA_span=(
            "ica_span_recovery",
            "mean",
        ),

        AA_MSE=("aa_mse", "mean"),
        PCA_MSE=("pca_mse", "mean"),
        ICA_MSE=("ica_mse", "mean"),

        ICA_convergence_rate=(
            "ica_converged",
            "mean",
        ),
        ICA_mean_iterations=(
            "ica_n_iter",
            "mean",
        ),
    )
)

display(compact)


## 8. Core figure: direct pattern recovery

In [ ]:

regime_order = [
    "PCA_control",
    "ICA_control",
    "AA_control",
    "metastable",
]

regime_labels = [
    "PCA-favorable",
    "ICA-favorable",
    "AA-favorable",
    "Metastable",
]

method_cols = [
    ("MS-AA", "aa_motif_recovery"),
    ("PCA", "pca_motif_recovery"),
    ("ICA", "ica_motif_recovery"),
]

x = np.arange(
    len(regime_order)
)

width = 0.24

fig, ax = plt.subplots(
    figsize=(9, 4.8)
)

for j, (label, col) in enumerate(
    method_cols
):
    means = []
    sems = []

    for regime in regime_order:
        vals = core.loc[
            core["regime"] == regime,
            col,
        ]

        means.append(vals.mean())
        sems.append(vals.sem())

    offset = (
        j - 1
    ) * width

    ax.bar(
        x + offset,
        means,
        width,
        yerr=sems,
        label=label,
        capsize=3,
    )

ax.set_xticks(x)
ax.set_xticklabels(regime_labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel(
    "One-to-one true-pattern recovery"
)
ax.set_title(
    "Method-calibrated latent-pattern recovery"
)
ax.legend()

plt.show()


## 9. Span-recovery control

In [ ]:

fig, ax = plt.subplots(
    figsize=(9, 4.8)
)

span_cols = [
    ("MS-AA", "aa_span_recovery"),
    ("PCA", "pca_span_recovery"),
    ("ICA", "ica_span_recovery"),
]

for j, (label, col) in enumerate(
    span_cols
):
    means = []
    sems = []

    for regime in regime_order:
        vals = core.loc[
            core["regime"] == regime,
            col,
        ]

        means.append(vals.mean())
        sems.append(vals.sem())

    offset = (
        j - 1
    ) * width

    ax.bar(
        x + offset,
        means,
        width,
        yerr=sems,
        label=label,
        capsize=3,
    )

ax.set_xticks(x)
ax.set_xticklabels(regime_labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel(
    "Recovery from full recovered span"
)
ax.set_title(
    "Pattern identity versus low-dimensional span"
)
ax.legend()

plt.show()


# PART B — Paired bootstrap and sign-flip inference

In [ ]:

PAIR_BOOT_REPS = 20000
PAIR_PERM_REPS = 20000
PAIR_RNG_SEED = 20260908


def bootstrap_mean_ci(
    values,
    n_boot=PAIR_BOOT_REPS,
    seed=PAIR_RNG_SEED,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    rng = np.random.default_rng(seed)

    idx = rng.integers(
        0,
        len(values),
        size=(
            n_boot,
            len(values),
        ),
    )

    boot_means = values[idx].mean(
        axis=1
    )

    lo, hi = np.quantile(
        boot_means,
        [0.025, 0.975],
    )

    return (
        float(values.mean()),
        float(lo),
        float(hi),
    )


def signflip_pvalue(
    values,
    n_perm=PAIR_PERM_REPS,
    seed=PAIR_RNG_SEED,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    observed = abs(
        values.mean()
    )

    rng = np.random.default_rng(seed)

    signs = rng.choice(
        [-1.0, 1.0],
        size=(
            n_perm,
            len(values),
        ),
    )

    perm_means = np.abs(
        (
            signs
            * values[None, :]
        ).mean(axis=1)
    )

    p = (
        np.sum(
            perm_means >= observed
        )
        + 1
    ) / (
        n_perm + 1
    )

    return float(p)


comparison_specs = [
    (
        "AA - PCA",
        "aa_motif_recovery",
        "pca_motif_recovery",
    ),
    (
        "AA - ICA",
        "aa_motif_recovery",
        "ica_motif_recovery",
    ),
    (
        "ICA - PCA",
        "ica_motif_recovery",
        "pca_motif_recovery",
    ),
]

paired_rows = []

for r_i, regime in enumerate(
    regime_order
):
    sub = (
        core[
            core["regime"] == regime
        ]
        .sort_values("rep")
        .copy()
    )

    for c_i, (
        comparison,
        col_a,
        col_b,
    ) in enumerate(comparison_specs):

        delta = (
            sub[col_a].to_numpy()
            - sub[col_b].to_numpy()
        )

        mean, lo, hi = (
            bootstrap_mean_ci(
                delta,
                seed=(
                    PAIR_RNG_SEED
                    + 100 * r_i
                    + c_i
                ),
            )
        )

        paired_rows.append({
            "regime": regime,
            "comparison": comparison,
            "n": len(delta),
            "mean_delta": mean,
            "CI_low": lo,
            "CI_high": hi,
            "signflip_p": (
                signflip_pvalue(
                    delta,
                    seed=(
                        PAIR_RNG_SEED
                        + 1000
                        + 100 * r_i
                        + c_i
                    ),
                )
            ),
            "fraction_positive": float(
                np.mean(delta > 0)
            ),
        })

paired_inference_df = pd.DataFrame(
    paired_rows
)

display(paired_inference_df)


## 10. Metastable pairwise method differences

In [ ]:

meta_pairs = paired_inference_df[
    paired_inference_df["regime"]
    == "metastable"
].copy()

display(meta_pairs)

fig, ax = plt.subplots(
    figsize=(7.5, 4.5)
)

x_meta = np.arange(
    len(meta_pairs)
)

means = meta_pairs[
    "mean_delta"
].to_numpy()

los = meta_pairs[
    "CI_low"
].to_numpy()

his = meta_pairs[
    "CI_high"
].to_numpy()

ax.errorbar(
    x_meta,
    means,
    yerr=np.vstack([
        means - los,
        his - means,
    ]),
    fmt="o",
    capsize=5,
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1,
)

ax.set_xticks(x_meta)
ax.set_xticklabels(
    meta_pairs["comparison"]
)
ax.set_ylabel(
    "Paired difference in motif recovery"
)
ax.set_title(
    "Brain-inspired metastable regime"
)

plt.show()


# PART C — Metastable robustness to noise

In [ ]:

results_df = load_results()

for noise_sd in NOISE_LEVELS:
    for rep in range(N_REPS_ROBUST):
        regime = "metastable_noise"
        condition = f"noise_{noise_sd}"

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    404,
                    int(noise_sd * 1000),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(
            seed,
            noise_sd=noise_sd,
            heterogeneity_scale=1.0,
        )

        diag = metastable_diagnostics(
            sim
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            extra={
                "condition": condition,
                "noise_sd": noise_sd,
                "mean_dwell": (
                    diag["mean_dwell"]
                ),
                "mean_activation_lag1": (
                    diag[
                        "mean_activation_lag1"
                    ]
                ),
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print("Noise robustness complete.")


# PART D — Metastable robustness to subject heterogeneity

In [ ]:

results_df = load_results()

for hetero in HETEROGENEITY_LEVELS:
    for rep in range(N_REPS_ROBUST):
        regime = "metastable_heterogeneity"
        condition = (
            f"heterogeneity_{hetero}"
        )

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    505,
                    int(hetero * 1000),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(
            seed,
            noise_sd=BASE_NOISE_SD,
            heterogeneity_scale=hetero,
        )

        diag = metastable_diagnostics(
            sim
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=N_TRUE_MOTIFS,
            seed=seed,
            regime=regime,
            rep=rep,
            extra={
                "condition": condition,
                "heterogeneity_scale": hetero,
                "mean_dwell": (
                    diag["mean_dwell"]
                ),
                "mean_activation_lag1": (
                    diag[
                        "mean_activation_lag1"
                    ]
                ),
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print("Heterogeneity robustness complete.")


# PART E — Metastable sensitivity to fitted K

In [ ]:

results_df = load_results()

for K_fit in K_LEVELS:
    for rep in range(N_REPS_ROBUST):
        regime = "metastable_K"
        condition = f"K_{K_fit}"

        if is_complete(
            results_df,
            regime,
            condition,
            rep,
        ):
            print(
                f"SKIP {condition} rep={rep}"
            )
            continue

        seed = int(
            np.random.SeedSequence(
                [
                    RNG_SEED,
                    606,
                    int(K_fit),
                    rep,
                ]
            ).generate_state(1)[0]
        )

        sim = simulate_metastable(seed)
        diag = metastable_diagnostics(
            sim
        )

        row = evaluate_methods(
            sim["subjects"],
            sim["true_patterns"],
            K_fit=K_fit,
            seed=seed,
            regime=regime,
            rep=rep,
            extra={
                "condition": condition,
                "mean_dwell": (
                    diag["mean_dwell"]
                ),
                "mean_activation_lag1": (
                    diag[
                        "mean_activation_lag1"
                    ]
                ),
            },
        )

        results_df = save_row(
            results_df,
            row,
        )

print("K sensitivity complete.")



## 11. K-sensitivity summary

For \(K < K_{\mathrm{true}}\), use the **coverage-adjusted** score as the primary quantity. The ordinary matched mean is also retained for diagnostic purposes.


In [ ]:

k_df = results_df[
    results_df["regime"]
    == "metastable_K"
].copy()

if len(k_df):
    k_summary = (
        k_df
        .groupby("K_fit")[
            [
                "aa_motif_recovery_coverage",
                "pca_motif_recovery_coverage",
                "ica_motif_recovery_coverage",
            ]
        ]
        .agg(["mean", "sem"])
    )

    display(k_summary)

    fig, ax = plt.subplots(
        figsize=(7.5, 4.5)
    )

    for label, col in [
        (
            "MS-AA",
            "aa_motif_recovery_coverage",
        ),
        (
            "PCA",
            "pca_motif_recovery_coverage",
        ),
        (
            "ICA",
            "ica_motif_recovery_coverage",
        ),
    ]:
        means = (
            k_df
            .groupby("K_fit")[col]
            .mean()
        )

        sems = (
            k_df
            .groupby("K_fit")[col]
            .sem()
        )

        ax.errorbar(
            means.index,
            means.values,
            yerr=sems.values,
            marker="o",
            label=label,
        )

    ax.axvline(
        N_TRUE_MOTIFS,
        linestyle="--",
        linewidth=1,
    )

    ax.set_xlabel("Fitted K")
    ax.set_ylabel(
        "Coverage-adjusted true-pattern recovery"
    )
    ax.set_title(
        "Metastable sensitivity to fitted model order"
    )
    ax.legend()

    plt.show()


# PART F — Representative metastable spatial patterns

In [ ]:

REPRESENTATIVE_SEED = 20260908

sim = simulate_metastable(
    REPRESENTATIVE_SEED
)

subjects, mu, sd = (
    normalize_across_subjects(
        sim["subjects"]
    )
)

true_patterns = (
    sim["true_patterns"] - mu
) / sd

X_stack = np.vstack(subjects)

# ---------- MS-AA ----------
msaa = fit_msaa(
    subjects,
    K=N_TRUE_MOTIFS,
    analysis_type="spatial",
    seed=REPRESENTATIVE_SEED,
    verbose=True,
)

S_group = np.mean(
    np.stack(
        [
            np.asarray(
                sub["S"],
                dtype=float,
            )
            for sub in msaa[
                "results_subj"
            ]
        ],
        axis=0,
    ),
    axis=0,
)

# ---------- PCA ----------
pca = PCA(
    n_components=N_TRUE_MOTIFS,
    random_state=REPRESENTATIVE_SEED,
).fit(X_stack)

pca_patterns = pca.components_

# ---------- ICA ----------
ica_fit = fit_fastica(
    X_stack,
    N_TRUE_MOTIFS,
    REPRESENTATIVE_SEED,
)

ica_patterns = (
    ica_fit["spatial_patterns"]
)

# Match patterns
aa_match = optimal_match(
    true_patterns,
    S_group,
    absolute=False,
)

pca_match = optimal_match(
    true_patterns,
    pca_patterns,
    absolute=True,
)

ica_match = optimal_match(
    true_patterns,
    ica_patterns,
    absolute=True,
)

AA_ORDERED = S_group[
    aa_match["recovered_idx"]
].copy()

PCA_ORDERED = pca_patterns[
    pca_match["recovered_idx"]
].copy()

ICA_ORDERED = ica_patterns[
    ica_match["recovered_idx"]
].copy()

# Orient PCA/ICA signs for visualization
for i in range(
    min(
        len(PCA_ORDERED),
        len(true_patterns),
    )
):
    if (
        np.corrcoef(
            true_patterns[i],
            PCA_ORDERED[i],
        )[0, 1]
        < 0
    ):
        PCA_ORDERED[i] *= -1

for i in range(
    min(
        len(ICA_ORDERED),
        len(true_patterns),
    )
):
    if (
        np.corrcoef(
            true_patterns[i],
            ICA_ORDERED[i],
        )[0, 1]
        < 0
    ):
        ICA_ORDERED[i] *= -1


plot_pattern_matrix(
    true_patterns,
    "True recurring network configurations",
    ylabel="True configuration",
)

plot_pattern_matrix(
    AA_ORDERED,
    "Matched MS-AA archetypes",
    ylabel="Matched archetype",
)

plot_pattern_matrix(
    PCA_ORDERED,
    "Matched PCA components",
    ylabel="Matched component",
)

plot_pattern_matrix(
    ICA_ORDERED,
    "Matched ICA spatial mixing patterns",
    ylabel="Matched ICA pattern",
)


# PART G — Final manuscript-oriented summary

In [ ]:

results_df = load_results()

core = results_df[
    results_df["regime"].isin(
        regime_order
    )
].copy()

final_rows = []

for regime in regime_order:
    sub = core[
        core["regime"] == regime
    ]

    if len(sub) == 0:
        continue

    final_rows.append({
        "regime": regime,
        "n": len(sub),

        "MS-AA motif recovery": (
            sub[
                "aa_motif_recovery"
            ].mean()
        ),
        "PCA motif recovery": (
            sub[
                "pca_motif_recovery"
            ].mean()
        ),
        "ICA motif recovery": (
            sub[
                "ica_motif_recovery"
            ].mean()
        ),

        "MS-AA subnetwork recovery": (
            sub[
                "aa_subnetwork_recovery"
            ].mean()
        ),
        "PCA subnetwork recovery": (
            sub[
                "pca_subnetwork_recovery"
            ].mean()
        ),
        "ICA subnetwork recovery": (
            sub[
                "ica_subnetwork_recovery"
            ].mean()
        ),

        "MS-AA span recovery": (
            sub[
                "aa_span_recovery"
            ].mean()
        ),
        "PCA span recovery": (
            sub[
                "pca_span_recovery"
            ].mean()
        ),
        "ICA span recovery": (
            sub[
                "ica_span_recovery"
            ].mean()
        ),

        "MS-AA MSE": (
            sub["aa_mse"].mean()
        ),
        "PCA MSE": (
            sub["pca_mse"].mean()
        ),
        "ICA MSE": (
            sub["ica_mse"].mean()
        ),

        "ICA convergence rate": (
            sub[
                "ica_converged"
            ].mean()
        ),
    })

final_summary_df = pd.DataFrame(
    final_rows
)

display(final_summary_df)



# Interpretation guide

A well-calibrated benchmark should show method-specific reversals rather than universal superiority.

### PCA-favorable control
PCA should recover the true covariance axes best. The use of distinct Gaussian source variances makes those axes identifiable.

### ICA-favorable control
ICA should most directly recover the true nonorthogonal mixing patterns generated by independent non-Gaussian sources.

### AA-favorable control
MS-AA should most directly recover the true extremal motifs.

### Brain-inspired metastable regime
This is the key scientific comparison. It is not generated exactly by PCA, ICA, or AA.

The most informative outcomes are:

1. **direct motif recovery** — does an individual recovered dimension correspond to an underlying network configuration?
2. **subnetwork recovery** — does it recover the configuration's network composition?
3. **span recovery** — was the latent information retained even if the individual basis vectors were rotated or mixed?
4. **reconstruction error** — how well was observed variance reconstructed?

The strongest archetype result would therefore *not* require MS-AA to beat PCA or ICA on reconstruction. Instead, it would show that under metastable network dynamics, MS-AA more directly identifies the recurring latent network configurations while PCA and/or ICA may retain the same information within a different low-dimensional basis.

The paired bootstrap section should be used to quantify MS-AA versus both PCA and ICA in the metastable regime.

---

## Important caution for the manuscript

This simulation is evidence about **representational geometry**, not proof that real brain activity is generated by archetypes. The metastable regime was deliberately designed to avoid making that assumption.

---

## ICA convergence note

ICA recovery values are shown for every simulated dataset. A convergence warning is retained as a diagnostic flag rather than used to exclude the estimate. This is particularly relevant in the PCA-favorable Gaussian control, where ICA's independent-component solution is not uniquely identifiable.

For manuscript reporting, convergence rate should be stated alongside the ICA results, but the recovery values should remain visible so that the benchmark contains all three methods in every regime.
